# FMVA v0.5 — Geneformer contextual gene embeddings

GPU notebook. It processes the exact cells selected by the frozen protocol, preserves
sample and lesion context, exports pretrained and shape-matched random representations, and never
reads future clinical labels. The run is resumable at sample-context granularity.

In [ ]:
# ruff: noqa: E402
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title 1. Install the frozen FMVA v0.5 alpha runtime
import subprocess
import sys
from pathlib import Path

DRIVE_NOTEBOOK_DIRS = [
    Path(
        "/content/drive/MyDrive/projects_bioinfo_2026/fm-value-audit-real/temporal_benchmark/08_notebooks"
    ),
    Path("/content/drive/MyDrive/fm-value-audit-real/temporal_benchmark/08_notebooks"),
]
wheel_candidates = []
for directory in DRIVE_NOTEBOOK_DIRS:
    if directory.is_dir():
        wheel_candidates.extend(directory.glob("fm_value_audit-0.5.0a1-py3-none-any.whl"))
if len(wheel_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one fm_value_audit-0.5.0a1 wheel in temporal_benchmark/08_notebooks; "
        f"found {wheel_candidates}"
    )
WHEEL = wheel_candidates[0]
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "anndata>=0.11.4,<0.12",
        "h5py>=3.11,<4",
        "safetensors>=0.4",
        "pyarrow>=16",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(WHEEL)], check=True)
print("Installed:", WHEEL)

In [ ]:
# ruff: noqa: E402
# @title 2. Mount Drive and locate the canonical project
from google.colab import drive

drive.mount("/content/drive")

import json
import os
from datetime import UTC, datetime
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch

PROJECT_CANDIDATES = [
    Path("/content/drive/MyDrive/projects_bioinfo_2026/fm-value-audit-real"),
    Path("/content/drive/MyDrive/fm-value-audit-real"),
]
PROJECT = next((path for path in PROJECT_CANDIDATES if path.is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError(f"Could not locate fm-value-audit-real; checked {PROJECT_CANDIDATES}")

TEMPORAL_V05 = PROJECT / "temporal_benchmark_v0.5"
PROTOCOL_DIR = TEMPORAL_V05 / "00_protocol"
CONTEXTUAL_DIR = TEMPORAL_V05 / "04_contextual_embeddings"
PROTOCOL_PATH = PROTOCOL_DIR / "FMVA_CONTEXTUAL_PROSPECTIVE_v0.5.json"
PREFLIGHT_PATH = PROTOCOL_DIR / "PREFLIGHT_MANIFEST.json"

STAGE1_CANDIDATES = [
    PROJECT / "outputs" / "stage1_full_cells_16gb",
    PROJECT / "outputs" / "stage1_full_cells",
]
STAGE1 = next((path for path in STAGE1_CANDIDATES if path.is_dir()), None)
if STAGE1 is None:
    raise FileNotFoundError(
        f"Could not locate full-cell Stage 1 outputs; checked {STAGE1_CANDIDATES}"
    )

MODEL_ROOT = PROJECT / "models"
PATHS = {
    "gse115978_h5ad": STAGE1 / "GSE115978_ALL_7186_raw_counts.h5ad",
    "gse179994_h5ad": STAGE1 / "GSE179994_ALL_150849_raw_counts.h5ad",
    "gse179994_lesion_mapping": STAGE1 / "GSE179994_official_lesion_sample_mapping.csv",
    "geneformer_model": MODEL_ROOT / "Geneformer-V1-10M" / "model.safetensors",
    "geneformer_symbol_map": MODEL_ROOT / "gene_dictionaries_30m" / "gene_name_id_dict_gc30M.pkl",
    "geneformer_tokens": MODEL_ROOT / "gene_dictionaries_30m" / "token_dictionary_gc30M.pkl",
    "geneformer_medians": MODEL_ROOT / "gene_dictionaries_30m" / "gene_median_dictionary_gc30M.pkl",
    "scgpt_model": MODEL_ROOT / "scGPT-whole-human" / "best_model.pt",
    "scgpt_vocab": MODEL_ROOT / "scGPT-whole-human" / "vocab.json",
    "scgpt_args": MODEL_ROOT / "scGPT-whole-human" / "args.json",
}
missing = [str(path) for path in PATHS.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")

print("Project:", PROJECT)
print("Stage 1:", STAGE1)
print(
    "CUDA:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
)

In [ ]:
# @title 3. Load the frozen protocol, preflight and model
import pickle
from importlib.metadata import version

from scipy import sparse

from fmva.contextual import (
    default_contextual_protocol,
    sha256_file,
    validate_contextual_artifact,
    write_contextual_artifact,
)
from fmva.contextual_workflow import (
    CONTEXT_COLUMNS,
    TokenGeneAccumulator,
    build_context_metadata,
    reduce_sample_context_partials,
    safe_partial_name,
    select_protocol_cells,
    write_sample_context_partial,
)
from fmva.model_runtime import (
    encode_geneformer_contextual_batch,
    load_geneformer_contextual_states,
    tokenize_geneformer_contextual,
)

if not (PROTOCOL_DIR / "V05_PREFLIGHT_COMPLETE.txt").is_file() or not PREFLIGHT_PATH.is_file():
    raise FileNotFoundError("Run FMVA_v05_00_preflight_protocol.ipynb first")
preflight = json.loads(PREFLIGHT_PATH.read_text(encoding="utf-8"))
protocol = default_contextual_protocol()
if sha256_file(PROTOCOL_PATH) != preflight["protocol_sha256"]:
    raise RuntimeError("Protocol hash no longer matches preflight")

DEVICE = "cuda" if torch.cuda.is_available() else None
if DEVICE is None:
    raise RuntimeError("A GPU runtime is required for this notebook")
SEQUENCE_LENGTH = 512
BATCH_SIZE = 4
MODEL_OUTPUT = CONTEXTUAL_DIR / "geneformer"
PARTIAL_ROOT = MODEL_OUTPUT / "sample_context_partials"
MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)
PARTIAL_ROOT.mkdir(parents=True, exist_ok=True)

with PATHS["geneformer_symbol_map"].open("rb") as handle:
    SYMBOL_MAP = pickle.load(handle)
with PATHS["geneformer_tokens"].open("rb") as handle:
    TOKEN_DICT = pickle.load(handle)
with PATHS["geneformer_medians"].open("rb") as handle:
    GENE_MEDIANS = pickle.load(handle)
PRETRAINED_STATE, RANDOM_STATE = load_geneformer_contextual_states(
    PATHS["geneformer_model"], seed=protocol.sampling.seed, device=DEVICE
)
MODEL_ID = "geneformer_contextual"
CONTROL_ID = "geneformer_random_contextual"
CHECKPOINT_KEY = "geneformer_model"
WIDTH = 256
print("Device:", DEVICE)
print("Sequence length:", SEQUENCE_LENGTH, "batch size:", BATCH_SIZE)

In [ ]:
# @title 4. Process one cohort with resumable sample-context checkpoints


def dense_rows(adata: ad.AnnData, positions: np.ndarray) -> np.ndarray:
    values = adata[positions, :].X
    if sparse.issparse(values):
        values = values.toarray()
    return np.asarray(values, dtype=np.float32)


def process_cohort(cohort: str, h5ad_path: Path, lesion_mapping: pd.DataFrame | None):
    cohort_partial_root = PARTIAL_ROOT / cohort
    cohort_partial_root.mkdir(parents=True, exist_ok=True)
    adata = ad.read_h5ad(h5ad_path, backed="r")
    try:
        context_metadata = build_context_metadata(
            adata.obs,
            cohort=cohort,
            official_lesion_mapping=lesion_mapping,
        )
        selected = select_protocol_cells(
            context_metadata,
            maximum_cells_per_sample_context=protocol.sampling.maximum_cells_per_sample_context,
            minimum_cells_per_sample_context=protocol.sampling.minimum_cells_per_sample_context,
            seed=protocol.sampling.seed,
        )
        selected["context_id"] = selected[CONTEXT_COLUMNS].astype(str).agg("|".join, axis=1)
        obs_index = pd.Index(adata.obs_names.astype(str))
        selected["matrix_row"] = obs_index.get_indexer(selected["cell_id"].astype(str))
        if (selected["matrix_row"] < 0).any():
            raise RuntimeError(f"{cohort}: selected cell IDs not found in H5AD")
        genes = [str(value) for value in adata.var_names]
        group_columns = ["sample_id", *CONTEXT_COLUMNS, "context_id"]
        groups = list(selected.groupby(group_columns, sort=True, dropna=False))
        print(cohort, "selected cells:", len(selected), "sample-contexts:", len(groups))

        for group_number, (group_key, group) in enumerate(groups, start=1):
            sample_id = str(group_key[0])
            context_id = str(group_key[-1])
            partial_path = cohort_partial_root / safe_partial_name(sample_id, context_id)
            if partial_path.is_file():
                continue
            accumulator = TokenGeneAccumulator.create(len(genes), WIDTH)
            positions = group["matrix_row"].to_numpy(dtype=int)
            for start in range(0, len(positions), BATCH_SIZE):
                batch_positions = positions[start : start + BATCH_SIZE]
                counts = dense_rows(adata, batch_positions)
                tokens, mask, gene_indices, _coverage = tokenize_geneformer_contextual(
                    genes, counts, SYMBOL_MAP, TOKEN_DICT, GENE_MEDIANS, SEQUENCE_LENGTH
                )
                pretrained_hidden = encode_geneformer_contextual_batch(
                    tokens, mask, PRETRAINED_STATE, device=DEVICE
                )
                random_hidden = encode_geneformer_contextual_batch(
                    tokens, mask, RANDOM_STATE, device=DEVICE
                )
                accumulator.update(gene_indices, pretrained_hidden, random_hidden)
                del counts, pretrained_hidden, random_hidden
                torch.cuda.empty_cache()
            gene_indices_final, pretrained, random, cell_counts = accumulator.finalise()
            metadata = {
                "sample_id": sample_id,
                "disease": str(group_key[1]),
                "cell_compartment": str(group_key[2]),
                "treatment_state": str(group_key[3]),
                "response_state": str(group_key[4]),
                "context_id": context_id,
                "selected_cells": str(len(group)),
            }
            temporary = partial_path.with_suffix(".tmp.npz")
            write_sample_context_partial(
                temporary,
                gene_indices=gene_indices_final,
                pretrained=pretrained,
                random=random,
                cell_counts=cell_counts,
                metadata=metadata,
            )
            os.replace(temporary, partial_path)
            if group_number % 10 == 0 or group_number == len(groups):
                print(cohort, f"{group_number}/{len(groups)} sample-contexts complete")
        partial_paths = sorted(cohort_partial_root.glob("sample_context_*.npz"))
        index, pretrained, random = reduce_sample_context_partials(
            partial_paths,
            gene_symbols=genes,
            width=WIDTH,
        )
        return index, pretrained, random, selected
    finally:
        adata.file.close()

In [ ]:
# @title 5. Run melanoma and NSCLC cohorts
lesion_mapping = pd.read_csv(PATHS["gse179994_lesion_mapping"])
cohort_outputs = []
for cohort, h5ad_path, mapping in [
    ("GSE115978", PATHS["gse115978_h5ad"], None),
    ("GSE179994", PATHS["gse179994_h5ad"], lesion_mapping),
]:
    index, pretrained, random, selected = process_cohort(cohort, h5ad_path, mapping)
    cohort_outputs.append((index, pretrained, random, selected))
    print(cohort, index.shape, pretrained.shape)

combined_index = pd.concat([item[0] for item in cohort_outputs], ignore_index=True)
combined_pretrained = np.vstack([item[1] for item in cohort_outputs])
combined_random = np.vstack([item[2] for item in cohort_outputs])
selected_cells = int(sum(len(item[3]) for item in cohort_outputs))
represented_samples = int(sum(item[3]["sample_id"].nunique() for item in cohort_outputs))
print("Combined:", combined_index.shape, combined_pretrained.shape)

In [ ]:
# @title 6. Write, hash and validate the contextual artifact
manifest = write_contextual_artifact(
    MODEL_OUTPUT,
    protocol_path=PROTOCOL_PATH,
    model_id=MODEL_ID,
    control_model_id=CONTROL_ID,
    checkpoint_sha256=preflight["file_hashes"][CHECKPOINT_KEY],
    input_hashes={
        "gse115978_h5ad": preflight["file_hashes"]["gse115978_h5ad"],
        "gse179994_h5ad": preflight["file_hashes"]["gse179994_h5ad"],
        "gse179994_lesion_mapping": preflight["file_hashes"]["gse179994_lesion_mapping"],
    },
    index=combined_index,
    pretrained=combined_pretrained,
    random=combined_random,
    selected_cells=selected_cells,
    represented_samples=represented_samples,
    sequence_length=SEQUENCE_LENGTH,
    sampling_seed=protocol.sampling.seed,
    pooling="token-to-gene mean within sample-context, then equal-weight sample mean",
    upstream_versions={
        "anndata": version("anndata"),
        "numpy": version("numpy"),
        "pandas": version("pandas"),
        "torch": version("torch"),
    },
)
audit = validate_contextual_artifact(MODEL_OUTPUT, protocol_path=PROTOCOL_PATH)
(MODEL_OUTPUT / "CONTEXTUAL_AUDIT.json").write_text(
    json.dumps(audit.model_dump(mode="json"), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
if audit.status == "FAIL":
    raise RuntimeError(audit.errors)
(MODEL_OUTPUT / "CONTEXTUAL_COMPLETE.txt").write_text(
    f"{MODEL_ID} complete at {datetime.now(UTC).isoformat()}\n",
    encoding="utf-8",
)
print(json.dumps(manifest.model_dump(mode="json"), indent=2))
print("AUDIT:", audit.status)